<p align = "center" draggable=”false” ><img src="https://github.com/AI-Maker-Space/LLM-Dev-101/assets/37101144/d1343317-fa2f-41e1-8af1-1dbb18399719" 
     width="200px"
     height="auto"/>
</p>

<h1 align="center" id="heading">OpenAI Agents SDK - AIM</h1>

In this notebook, we'll go over some of the key features of the OpenAI Agents SDK - as explored through a notebook-ified version of their [Research Bot](https://github.com/openai/openai-agents-python/tree/main/examples/research_bot).

In [1]:
### You don't need to run this cell if you're running this notebook locally. 

#!pip install -qU openai-agents

API Key:

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# The OPENAI_API_KEY is now loaded from the .env file
print("Environment variables loaded from .env file")

Environment variables loaded from .env file


Nest Async:

In [3]:
import nest_asyncio
nest_asyncio.apply()

## Agents

As may be expected, the primary thing we'll do in the Agents SDK is construct Agents!

Agents are constructed with a few basic properties:

- A prompt, which OpenAI is using the language "instruction" for, that determines the behaviour or goal of the Agent
- A model, the "brain" of the Agent

They also typically include an additional property: 

- Tool(s) that equip the Agent with things it can use to get stuff done

### Task 1: Create Planner Agent

Let's start by creating our "Planner Agent" - which will come up with the initial set of search terms that should answer a query provided by the user. 



In [4]:
from pydantic import BaseModel
from agents import Agent

PLANNER_PROMPT = (
    "You are a helpful research assistant. Given a query, come up with a set of web searches to perform" 
    "to best answer the query. Output between 5 and 20 terms to query for."
)

Next, we'll define the data models that our Planner Agent will use to structure its output. We'll create:

1. `WebSearchItem` - A model for individual search items, containing the search query and reasoning
2. `WebSearchPlan` - A container model that holds a list of search items

These Pydantic models will help ensure our agent returns structured data that we can easily process.


In [5]:
class WebSearchItem(BaseModel):
    reason: str
    "Your reasoning for why this search is important to the query."

    query: str
    "The search term to use for the web search."

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]
    """A list of web searches to perform to best answer the query."""

Now we'll create our Planner Agent using the Agent class from the OpenAI Agents SDK. This agent will use the instructions defined in `PLANNER_PROMPT` and will output structured data in the form of our WebSearchPlan model. We're using the GPT-4o model for this agent to ensure high-quality search term generation.

> NOTE: When we provide an `output_type` - the model will return a [structured response](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses).


In [6]:
planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_PROMPT,
    model="gpt-4.1",
    output_type=WebSearchPlan,
)

#### ❓Question #1:

Why is it important to provide a structured response template? (As in: Why are structured outputs helpful/preferred in Agentic workflows?)

##### ✅ Answer:
- Structured outputs define template JSON schemas that model outputs can reliably adhere to 
- Without these, the model output will not be consistent or reliable
- Tool integrarations become more effective with defined input/output formats
- This improves error handling via validation of the structured outputs
- This also enables integration with other traditional API based systems


### Task 2: Create Search Agent

Now we'll create our Search Agent, which will be responsible for executing web searches based on the terms generated by the Planner Agent. This agent will take each search query, perform a web search using the `WebSearchTool`, and then summarize the results in a concise format.

> NOTE: We are using the `WebSearchTool`, a hosted tool that can be used as part of an `OpenAIResponsesModel` as outlined in the [documentation](https://openai.github.io/openai-agents-python/tools/). This is based on the tools available through OpenAI's new [Responses API](https://openai.com/index/new-tools-for-building-agents/).

The `SEARCH_PROMPT` below instructs the agent to create brief, focused summaries of search results. These summaries are designed to be 2-3 paragraphs, under 300 words, and capture only the essential information without unnecessary details. The goal is to provide the Writer Agent with clear, distilled information that can be efficiently synthesized into the final report.


In [7]:
SEARCH_PROMPT = (
    "You are a research assistant. Given a search term, you search the web for that term and"
    "produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300"
    "words. Capture the main points. Write succinctly, no need to have complete sentences or good"
    "grammar. This will be consumed by someone synthesizing a report, so its vital you capture the"
    "essence and ignore any fluff. Do not include any additional commentary other than the summary"
    "itself."
)

Now we'll create our Search Agent using the Agent class from the OpenAI Agents SDK. This agent will use the instructions defined in `SEARCH_PROMPT` and will utilize the `WebSearchTool` to perform web searches. We're configuring it with `tool_choice="required"` to ensure it always uses the search tool when processing requests.

> NOTE: We can, as demonstrated, indicate how we want our model to use tools. You can read more about that at the bottom of the page [here](https://openai.github.io/openai-agents-python/agents/)

In [8]:
from agents import WebSearchTool
from agents.model_settings import ModelSettings

search_agent = Agent(
    name="Search agent",
    instructions=SEARCH_PROMPT,
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required"),
)

#### ❓ Question #2: 

What other tools are supported in OpenAI's Responses API?

##### ✅ Answer:
Response API supports these built-in tools:
- Web Search: Allows models to search web for latest info before responding
- File Search: Allows models to serach file for relevant info before responding
- Computer Use: Build a computer using agent that can perform tasks on user's behalf
- Image Generation: Allows models to generate or edit images
- Code Interpreter: Allows models to write and run Python code
- Function Calling: To allows custom tool integration
- Remote MCP servers: Allows access to new capabilities exposed via MCP servers

### Task 3: Create Writer Agent

Finally, we'll create our Writer Agent, which will synthesize all the research findings into a comprehensive report. This agent takes the original query and the research summaries from the Search Agent, then produces a structured report with follow-up questions.

The Writer Agent will:
1. Create an outline for the report structure
2. Generate a detailed markdown report (5-10 pages)
3. Provide follow-up questions for further research

We'll define the prompt for this agent in the next cell. This prompt will instruct the Writer Agent on how to synthesize research findings into a comprehensive report with follow-up questions.

In [9]:
# Original research report prompt (commented out)
# WRITER_PROMPT = (
#     "You are a senior researcher tasked with writing a cohesive report for a research query. "
#     "You will be provided with the original query, and some initial research done by a research "
#     "assistant.\n"
#     "You should first come up with an outline for the report that describes the structure and "
#     "flow of the report. Then, generate the report and return that as your final output.\n"
#     "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
#     "for 5-10 pages of content, at least 1000 words.\n"
#     "For the follow-up questions, provide exactly 5 unique questions that would help extend "
#     "this research. Do not repeat questions."
# )

# New Developer Blog Writer Prompt
WRITER_PROMPT = (
    "You are an expert developer and technical writer who creates educational blog posts for developers. "
    "Your goal is to teach complex technical concepts in an accessible, engaging way.\n\n"
    
    "You will be provided with a technical topic and research findings. Your task is to create a comprehensive "
    "blog post that:\n\n"
    
    "1. **Educational Focus**: Break down complex concepts into digestible sections\n"
    "2. **Practical Examples**: Include real-world code examples and use cases\n"
    "3. **Progressive Learning**: Start with basics and build up to advanced concepts\n"
    "4. **Developer-Friendly**: Use clear explanations, analogies, and visual descriptions\n"
    "5. **Actionable Content**: Provide practical takeaways and next steps\n\n"
    
    "Structure your blog post with:\n"
    "- **Introduction**: Hook the reader and explain why this topic matters\n"
    "- **Core Concepts**: Break down the main ideas with examples\n"
    "- **Implementation**: Show practical code examples\n"
    "- **Best Practices**: Share tips and common pitfalls to avoid\n"
    "- **Conclusion**: Summarize key points and suggest next steps\n\n"
    
    "Write in a conversational, engaging tone that makes technical content approachable. "
    "Use analogies and real-world examples to explain abstract concepts. "
    "Include code snippets with explanations and comments.\n\n"
    
    "For follow-up topics, suggest 5 related concepts that would help readers deepen their understanding "
    "or explore adjacent technologies."
)

### 🎯 Developer Blog Writer Agent

This agent has been specifically designed to create educational blog posts for developers. Unlike the original research report writer, this agent focuses on:

**Key Features:**
- **Educational Content**: Breaks down complex technical concepts into digestible sections
- **Code Examples**: Includes practical, real-world code snippets with explanations
- **Progressive Learning**: Starts with basics and builds up to advanced concepts
- **Developer-Friendly**: Uses clear explanations, analogies, and visual descriptions
- **Actionable Content**: Provides practical takeaways and next steps

**Blog Post Structure:**
1. **Introduction** - Hook the reader and explain why the topic matters
2. **Core Concepts** - Break down main ideas with examples
3. **Implementation** - Show practical code examples
4. **Best Practices** - Share tips and common pitfalls
5. **Conclusion** - Summarize key points and suggest next steps

**Perfect for topics like:**
- New programming languages or frameworks
- Software architecture patterns
- DevOps and deployment strategies
- API design and integration
- Database optimization techniques
- Security best practices
- Performance optimization


#### 🏗️ Activity #1: 

This prompt is quite generic - modify this prompt to produce a report that is more personalized to either your personal preference, or more appropriate for a specific use case (eg. law domain research)

Now we'll create our Writer Agent using the Agent class from the OpenAI Agents SDK. This agent will synthesize all the research findings into a comprehensive report. We're configuring it with the `ReportData` output type to structure the response with a short summary, markdown report, and follow-up questions.

In [10]:
class BlogPostData(BaseModel):
    short_summary: str
    """A brief 2-3 sentence summary of what developers will learn from this blog post."""

    markdown_blog_post: str
    """The complete blog post in markdown format with code examples and explanations."""

    follow_up_topics: list[str]
    """5 related technical topics that would help readers deepen their understanding or explore adjacent technologies."""

Now we'll define our Writer Agent using the Agent class from the OpenAI Agents SDK. This agent will take the original query and research summaries, then synthesize them into a comprehensive report with follow-up questions. We've defined a custom output type called `ReportData` that structures the response with a short summary, markdown report, and follow-up questions.

In [11]:
writer_agent = Agent(
    name="DeveloperBlogWriter",
    instructions=WRITER_PROMPT,
    model="o3-mini",
    output_type=BlogPostData,
)

#### ❓ Question #3: 

Why are we electing to use a reasoning model for writing our report?

##### ✅ Answer:
- The report writing taks requires complex, multi-step reasoning, and planning
- Reasoning models can think step by step methodically, showing the thinking steps
- They can maintain logical and lexical coherence


## Task 4: Create Utility Classes 

We'll define utility classes to help with displaying progress and managing the research workflow. The Printer class below will provide real-time updates on the research process.


The Printer class provides real-time progress updates during the research process. It uses Rich's Live display to show dynamic content with spinners for in-progress items and checkmarks for completed tasks. The class maintains a dictionary of items with their completion status and can selectively hide checkmarks for specific items. This creates a clean, interactive console experience that keeps the user informed about the current state of the research workflow.

In [12]:
from typing import Any

from rich.console import Console, Group
from rich.live import Live
from rich.spinner import Spinner

class Printer:
    def __init__(self, console: Console):
        self.live = Live(console=console)
        self.items: dict[str, tuple[str, bool]] = {}
        self.hide_done_ids: set[str] = set()
        self.live.start()

    def end(self) -> None:
        self.live.stop()

    def hide_done_checkmark(self, item_id: str) -> None:
        self.hide_done_ids.add(item_id)

    def update_item(
        self, item_id: str, content: str, is_done: bool = False, hide_checkmark: bool = False
    ) -> None:
        self.items[item_id] = (content, is_done)
        if hide_checkmark:
            self.hide_done_ids.add(item_id)
        self.flush()

    def mark_item_done(self, item_id: str) -> None:
        self.items[item_id] = (self.items[item_id][0], True)
        self.flush()

    def flush(self) -> None:
        renderables: list[Any] = []
        for item_id, (content, is_done) in self.items.items():
            if is_done:
                prefix = "✅ " if item_id not in self.hide_done_ids else ""
                renderables.append(prefix + content)
            else:
                renderables.append(Spinner("dots", text=content))
        self.live.update(Group(*renderables))

Let's create a ResearchManager class that will orchestrate the research process. This class will:
1. Plan searches based on the query
2. Perform those searches to gather information
3. Write a comprehensive report based on the gathered information
4. Display progress using our Printer class


In [13]:
from __future__ import annotations

import asyncio
import time

from agents import Runner, custom_span, gen_trace_id, trace
from rich.console import Console

class ResearchManager:
    def __init__(self):
        self.console = Console()
        self.printer = Printer(self.console)

    async def run(self, query: str) -> None:
        trace_id = gen_trace_id()
        with trace("Research trace", trace_id=trace_id):
            self.printer.update_item(
                "trace_id",
                f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}",
                is_done=True,
                hide_checkmark=True,
            )

            self.printer.update_item(
                "starting",
                "Starting research...",
                is_done=True,
                hide_checkmark=True,
            )
            search_plan = await self._plan_searches(query)
            search_results = await self._perform_searches(search_plan)
            blog_post = await self._write_blog_post(query, search_results)

            final_summary = f"Blog post summary\n\n{blog_post.short_summary}"
            self.printer.update_item("final_blog_post", final_summary, is_done=True)

            self.printer.end()

        print("\n\n=====BLOG POST=====\n\n")
        print(f"Blog Post: {blog_post.markdown_blog_post}")
        print("\n\n=====FOLLOW UP TOPICS=====\n\n")
        unique_topics = []
        seen = set()
        
        for topic in blog_post.follow_up_topics:
            if topic not in seen:
                unique_topics.append(topic)
                seen.add(topic)
        
        for i, topic in enumerate(unique_topics, 1):
            print(f"{i}. {topic}")

    async def _plan_searches(self, query: str) -> WebSearchPlan:
        self.printer.update_item("planning", "Planning searches...")
        result = await Runner.run(
            planner_agent,
            f"Query: {query}",
        )
        self.printer.update_item(
            "planning",
            f"Will perform {len(result.final_output.searches)} searches",
            is_done=True,
        )
        return result.final_output_as(WebSearchPlan)

    async def _perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        with custom_span("Search the web"):
            self.printer.update_item("searching", "Searching...")
            num_completed = 0
            max_concurrent = 5
            results = []
            
            for i in range(0, len(search_plan.searches), max_concurrent):
                batch = search_plan.searches[i:i+max_concurrent]
                tasks = [asyncio.create_task(self._search(item)) for item in batch]
                
                for task in asyncio.as_completed(tasks):
                    try:
                        result = await task
                        if result is not None:
                            results.append(result)
                    except Exception as e:
                        print(f"Search error: {e}")
                        
                    num_completed += 1
                    self.printer.update_item(
                        "searching", f"Searching... {num_completed}/{len(search_plan.searches)} completed"
                    )
            
            self.printer.mark_item_done("searching")
            return results

    async def _search(self, item: WebSearchItem) -> str | None:
        input = f"Search term: {item.query}\nReason for searching: {item.reason}"
        try:
            result = await Runner.run(
                search_agent,
                input,
            )
            return str(result.final_output)
        except Exception as e:
            print(f"Error searching for '{item.query}': {e}")
            return None

    async def _write_blog_post(self, query: str, search_results: list[str]) -> BlogPostData:
        self.printer.update_item("writing", "Thinking about blog post...")
        input = f"Original query: {query}\nSummarized search results: {search_results}"
        
        result = Runner.run_streamed(
            writer_agent,
            input,
        )
        
        update_messages = [
            "Thinking about blog post...",
            "Planning blog post structure...",
            "Writing outline...",
            "Creating sections...",
            "Adding code examples...",
            "Finalizing blog post...",
            "Finishing blog post...",
        ]

        last_update = time.time()
        next_message = 0
        
        async for event in result.stream_events():
            if time.time() - last_update > 5 and next_message < len(update_messages):
                self.printer.update_item("writing", update_messages[next_message])
                next_message += 1
                last_update = time.time()
                
        self.printer.mark_item_done("writing")
        return result.final_output_as(BlogPostData)

#### 🏗️ Activity #2:

Convert the above flow into a flowchart style image (software of your choosing, but if you're not sure which to use try [Excallidraw](https://excalidraw.com/)) that outlines how the different Agents interact with each other. 

> HINT: Cursor's AI (CMD+L or CTRL+L on Windows) would be a helpful way to get a basic diagram that you can add more detail to!

## 🔄 Agent Interaction Flow

The following diagram shows how the different agents interact to create a developer blog post:




In [14]:
# High-resolution Mermaid diagram generation
from mermaid import Mermaid
from IPython.display import Image, display

# Use the corrected accurate diagram with verified model names
mermaid_diagram = """
graph TD
    A["User Query: Technical Topic"] --> B["ResearchManager<br/>(Orchestrator)"]
    
    B --> B1["1. _plan_searches()"]
    B1 --> C["Planner Agent<br/>(gpt-4.1)"]
    C --> D["WebSearchPlan<br/>List of search terms with reasoning"]
    D --> B2["2. _perform_searches()"]
    
    B2 --> E["Search Agent<br/>(with WebSearchTool)"]
    E --> F["WebSearchTool<br/>(Parallel Execution)"]
    F --> G1["Search Batch 1<br/>(5 concurrent)"]
    F --> G2["Search Batch 2<br/>(5 concurrent)"]
    F --> G3["Search Batch N<br/>(5 concurrent)"]
    
    G1 --> H["Search Results<br/>Summarized web content"]
    G2 --> H
    G3 --> H
    
    H --> B3["3. _write_blog_post()"]
    B3 --> I["Developer Blog Writer Agent<br/>(o3-mini)"]
    I --> J["BlogPostData<br/>- short_summary<br/>- markdown_blog_post<br/>- follow_up_topics"]
    J --> K["Final Output<br/>Educational Blog Post"]
    
    subgraph ProgressTracking["Progress Tracking"]
        P1["Printer: Planning searches..."]
        P2["Printer: Searching... X/Y completed"]
        P3["Printer: Thinking about blog post..."]
        P4["Printer: Finalizing blog post..."]
    end
    
    subgraph ErrorHandling["Error Handling"]
        E1["Search Error Handling<br/>- Try/catch per search<br/>- Continue on failure"]
        E2["Agent Error Handling<br/>- Graceful degradation"]
    end
    
    subgraph DataFlow["Data Models"]
        D1["WebSearchPlan<br/>- searches: list[WebSearchItem]<br/>- Each item has query + reason"]
        D2["BlogPostData<br/>- short_summary: str<br/>- markdown_blog_post: str<br/>- follow_up_topics: list[str]"]
    end
    
    B -.-> P1
    B1 -.-> P1
    B2 -.-> P2
    B3 -.-> P3
    B3 -.-> P4
    
    F -.-> E1
    C -.-> E2
    E -.-> E2
    I -.-> E2
    
    style A fill:#e1f5fe
    style B fill:#ffeb3b
    style K fill:#c8e6c9
    style C fill:#fff3e0
    style E fill:#fff3e0
    style I fill:#fff3e0
    style F fill:#f3e5f5
    style G1 fill:#e8f5e8
    style G2 fill:#e8f5e8
    style G3 fill:#e8f5e8
"""

print("🤖 Generating Verified High-Resolution Mermaid Diagram...")

# Create Mermaid instance and save as SVG
try:
    # Create Mermaid instance without resolution parameters
    mm = Mermaid(mermaid_diagram)
    
    # Save to SVG (better quality and scalable)
    mm.to_svg("agent_interaction_flow_verified.svg")
    
    print("✅ Verified high-resolution Mermaid diagram saved as 'agent_interaction_flow_verified.svg'")
    print("📐 SVG format provides better quality and scalability")
    
    # Display the saved SVG
    display(Image("agent_interaction_flow_verified.svg"))
    
except Exception as e:
    print(f"Error generating SVG: {e}")
    print("Falling back to HTML display...")
    
    # Fallback to HTML display
    from IPython.display import display, HTML
    display(HTML(f"""
    <div class="mermaid">
    {mermaid_diagram}
    </div>
    <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
    <script>mermaid.initialize({{startOnLoad:true}});</script>
    """))

🤖 Generating Verified High-Resolution Mermaid Diagram...
✅ Verified high-resolution Mermaid diagram saved as 'agent_interaction_flow_verified.svg'
📐 SVG format provides better quality and scalability
Error generating SVG: Cannot embed the 'svg' image format
Falling back to HTML display...


### 📋 Step-by-Step Process Flow

```
1. USER INPUT
    ↓
"Write a blog post about React Hooks"
RESEARCH MANAGER
    ↓
Coordinates the entire process
    ↓
Real-time progress updates:
"Planning searches..."
"Searching... X/Y completed"
"Thinking about blog post..."
"Finalizing blog post..."
PLANNER AGENT
    ↓
Creates WebSearchPlan with search terms:
"React Hooks tutorial"
"useState useEffect examples"
"React Hooks best practices"
"Custom hooks patterns"
"React Hooks vs class components"
SEARCH AGENT (Multiple parallel searches)
    ↓
Uses WebSearchTool to search web
    ↓
Returns summarized search results
DEVELOPER BLOG WRITER AGENT
    ↓
Synthesizes all research into BlogPostData:
short_summary: "Learn React Hooks fundamentals..."
markdown_blog_post: Complete educational blog post
follow_up_topics: ["Advanced React Patterns", "State Management", ...]
FINAL OUTPUT
    ↓
Educational blog post ready for publication
```

### 🔧 Key Components

**Agents:**
- **Planner Agent**: Strategic thinking, creates search plan
- **Search Agent**: Information gathering, web research
- **Developer Blog Writer Agent**: Educational content creation, technical writing with code examples

**Tools:**
- **WebSearchTool**: Performs actual web searches

**Data Models:**
- **WebSearchPlan**: Structured search strategy with list of WebSearchItem
- **WebSearchItem**: Individual search with query and reasoning
- **BlogPostData**: Final blog post content with summary, markdown, and follow-up topics

**Manager:**
- **ResearchManager**: Orchestrates the entire workflow

**Error Handling:**
- **Search Error Handling**: Try/catch per search, continue on failure
- **Agent Error Handling**: Graceful degradation for agent failures


## Task 5: Running Our Agent

Now let's run our agent! The main function below will prompt the user for a research topic, then pass that query to our ResearchManager to handle the entire research process. The ResearchManager will: 

1. Break down the query into search items
2. Search for information on each item
3. Write a comprehensive report based on the search results

Let's see it in action!

In [15]:
async def main() -> None:
    query = input("What would you like to research? ")
    await ResearchManager().run(query)

In [16]:
asyncio.run(main())

Output()



=====BLOG POST=====


Blog Post: # Building an Intelligent Agent with Tools Using the OpenAI Agents SDK

Imagine building an AI assistant that not only chats with users but can also search the web, execute code, or even manage files—all on its own! In this post, we'll walk through how to implement such an agent using the OpenAI Agents SDK. We'll demystify the core concepts, review practical examples, and wrap up with actionable tips to help you get started.

## Introduction

With the rise of automation and intelligent workflows, developers need tools that can extend the natural language capabilities of AI models. OpenAI's Agents SDK provides a framework where you can seamlessly integrate various tools such as hosted search, file management, code interpretation, and more into your agent. Whether you're working on a chatbot, automated assistant, or a data retrieval system, learning how to integrate these tools will significantly enhance your agent's abilities and streamline your workfl

---

## Sample Report in Markdown 

---



=====BLOG POST=====


Blog Post: # Building an Intelligent Agent with Tools Using the OpenAI Agents SDK

Imagine building an AI assistant that not only chats with users but can also search the web, execute code, or even manage files—all on its own! In this post, we'll walk through how to implement such an agent using the OpenAI Agents SDK. We'll demystify the core concepts, review practical examples, and wrap up with actionable tips to help you get started.

## Introduction

With the rise of automation and intelligent workflows, developers need tools that can extend the natural language capabilities of AI models. OpenAI's Agents SDK provides a framework where you can seamlessly integrate various tools such as hosted search, file management, code interpretation, and more into your agent. Whether you're working on a chatbot, automated assistant, or a data retrieval system, learning how to integrate these tools will significantly enhance your agent's abilities and streamline your workflow.

## Core Concepts

Before diving into the code, let's break down the main components of the Agents SDK:

### 1. Hosted Tools

These are tools that run alongside the language model on OpenAI's servers. Examples include:

- **Web Search:** Let your agent perform real-time online searches to retrieve current information.
- **File Search:** Query vector stores and file systems hosted by OpenAI.
- **Computer Use and Code Interpreter:** Execute code snippets in a controlled environment, which is useful for data analysis or generating on-the-fly computations.
- **Image Generation:** Create images from textual prompts for creative or illustrative purposes.

### 2. Function Tools

Function tools let you wrap your own local functions with a JSON schema. This means the agent can call your custom functions as if they were native tools. It’s a great way to add bespoke capabilities to your agent. 

### 3. Agents as Tools

Not only can you equip an agent with tools, but you can also expose an entire agent as a callable tool. This is particularly useful for building layered or domain-specialized agents that delegate tasks among themselves based on expertise.

### 4. Local MCP Servers

The Model Context Protocol (MCP) server allows your agent to interact with external systems running on your machine. This extends the agent's capabilities, making it possible to integrate with local databases or applications.

## Implementation

Let's walk through a simple example of how to implement an agent with a custom tool using the OpenAI Agents SDK. In our example, we'll create an agent that can perform a simple arithmetic operation using a function tool.

### Step 1: Set Up Your Environment

First, ensure you have Python 3.9+ installed. Then, set up a virtual environment and install the SDK:

```bash
python -m venv .venv
source .venv/bin/activate  # For Windows: .venv\Scripts\activate
pip install openai-agents
```

### Step 2: Define a Function Tool

We will create a simple function that performs addition. Decorate it with `@function_tool` (or appropriate decorator based on the SDK version) so it can be recognized by the agent.

```python
from openai_agents import Agent, function_tool

@function_tool
def add_numbers(a: float, b: float) -> float:
    """Adds two numbers and returns the result."""
    return a + b

# You may need to specify a JSON schema for the tool if required by the SDK version
```

### Step 3: Initialize and Configure Your Agent

Next, initialize your agent and add the custom tool:

```python
# Create an instance of the agent
agent = Agent(
    name="ArithmeticAgent",
    instructions="You have access to a tool that can add two numbers."
)

# Register the custom function tool with the agent
agent.add_tool(add_numbers)

# Optionally, add hosted tools or other types of tools as needed
```

### Step 4: Running the Agent

After setting up, you can run the agent. The agent will decide based on its instruction set when to call the tool.

```python
# Example input where the agent might decide to use the add_numbers tool
response = agent.run("Calculate the sum of 7 and 3.")
print(response)  # Expected output: 10 (or a message invoking the add_numbers tool)
```

The SDK abstracts away much of the complexity. You can observe tool calls, set up callbacks, and even trace execution to ensure your agent behaves as intended.

## Best Practices

Here are some tips to keep in mind when integrating tools with the Agents SDK:

- **Keep Tools Focused:** Each tool should have a single, well-defined responsibility. This improves clarity and reliability when the agent decides which tool to use.

- **Use Clear Descriptions:** Provide concise and clear descriptions for each tool. This helps the language model better understand the tool's purpose and how to use it.

- **Deploy Guardrails:** Implement input and output validation. Tools should validate parameters via schemas (e.g., using libraries like Zod) to minimize unexpected behavior and errors.

- **Avoid Infinite Loops:** The SDK supports configuration parameters like `tool_choice` and `resetToolChoice` to prevent the model from getting stuck in loops after a tool call.

- **Debug Logging:** For troubleshooting issues, enable debug logging by setting appropriate environment variables. This is especially useful when working with external APIs or when the agent's decision process becomes complex.

- **Experiment with Tool Combinations:** Leverage the SDK’s ability to chain tools; for example, combine a web search tool with a function tool to first retrieve data, then process it programmatically.

## Conclusion and Next Steps

In this post, we explored how to implement an agent with integrated tools using the OpenAI Agents SDK. We broke down core concepts, walked through a practical implementation example, and shared best practices to avoid common pitfalls. 

Now is a great time to experiment with the SDK by integrating additional tools such as web search or image generation. Here are some actionable next steps:

1. Explore the [OpenAI Agents SDK documentation](https://openai.github.io/openai-agents-js/guides/quickstart/) to understand more features.
2. Experiment with adding multiple tools and observe how the agent decides which one to use.
3. Try building a multi-agent system where one agent calls another as a tool for layered decision-making.
4. Integrate error handling and logging mechanisms to improve the reliability of your agent.
5. Join community forums and GitHub repositories to share your experience and learn from other developers.

Happy coding and may your agents be ever intelligent!


=====FOLLOW UP TOPICS=====


1. Advanced error handling in agent implementations
2. Integrating external APIs with function tools
3. Multi-agent workflows and inter-agent communication
4. Comparison between OpenAI Agents SDK and LangChain
5. Deploying AI agents with continuous integration and scaling strategies